# Аналитический отчёт по заявкам на обучение «НОРД-Класс»

Онлайн-платформа принимает заявки студентов на курсы, внешний оператор
присылает их ежедневным «сырым» CSV-файлом: в нём есть и корректные, и
ошибочные строки. Этот отчёт показывает, что находится в данных на момент
выгрузки: объём, качество, распределения по курсам и статусам и потенциальную
выручку по активным заявкам.

**Как запускать:** открыть ноутбук из корня проекта `edu_pipeline` и выполнить
`Run All` - ячейки идут строго по порядку и читают только файлы из
`data/final/`. Кодировка указана явно везде (`utf-8-sig`, у файлов есть BOM).

In [1]:
import csv
import re
from collections import Counter, defaultdict
from datetime import datetime
from pathlib import Path

DATA = Path("data/final")


def load_csv(name):
    """Читает CSV проекта; кодировка указана явно (файлы с BOM)."""
    with (DATA / name).open(encoding="utf-8-sig", newline="") as f:
        return [dict(row) for row in csv.DictReader(f)]


enrollments = load_csv("enrollments_raw.csv")
students = load_csv("students.csv")
courses = load_csv("courses.csv")
print(f"Заявки: {len(enrollments)} строк; студенты: {len(students)}; курсы: {len(courses)}")

Заявки: 19 строк; студенты: 16; курсы: 5


## Показатель 1. Объём сырых данных

Сколько строк прислали и сколько среди них реальных заявок: одинаковые номера
- это дубликаты, их считаем отдельно, потому что в работу должна попасть
заявка только один раз.

In [2]:
ids = Counter(r["enrollment_id"] for r in enrollments)
duplicates = [enrollment_id for enrollment_id, cnt in ids.items() if cnt > 1]
print(f"Всего строк в файле: {len(enrollments)}")
print(f"Уникальных номеров заявок: {len(ids)}")
print(f"Дубликаты: {duplicates}")

Всего строк в файле: 19
Уникальных номеров заявок: 18
Дубликаты: ['EDU-202610-0006']


**Интерпретация.** В выгрузке 19 строк, но уникальных заявок 18: заявка
EDU-202610-0006 пришла дважды. Значит, оператор в этот день отработал
дубликат дважды - при загрузке вторая копия должна быть отброшена, иначе
статистика и выручка задвоятся.

## Показатель 2. Качество данных: валидность строк

Проверяем каждое поле по простым правилам: номер вида EDU-ГГГГММ-NNNN с
настоящим месяцем, почта с доменной зоной, курс из справочника, известный
статус, дата, неотрицательная длительность, формат online/offline. Собираем
**все** причины отказа строки, а не только первую.

In [3]:
ID_RE = re.compile(r"EDU-\d{4}(?:0[1-9]|1[0-2])-\d{4}")
EMAIL_RE = re.compile(r"^[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}$")
COURSES_KNOWN = {c["course_code"] for c in courses}
STATUS_OK = {"new", "enrolled", "on_review", "completed", "cancelled"}


def check(row):
    """Возвращает список причин отказа строки; пустой список - строка валидна."""
    reasons = []
    if None in row:
        reasons.append("лишние колонки")
    if not ID_RE.fullmatch(row["enrollment_id"] or ""):
        reasons.append("неверный номер заявки")
    if not EMAIL_RE.fullmatch((row["student_email"] or "").strip()):
        reasons.append("неверный e-mail")
    if (row["course_code"] or "").strip() not in COURSES_KNOWN:
        reasons.append("курс не из справочника")
    if (row["status"] or "").strip() not in STATUS_OK:
        reasons.append("неизвестный статус")
    try:
        datetime.strptime(row["created_date"] or "", "%Y-%m-%d")
    except ValueError:
        reasons.append("неверная дата")
    try:
        if float(row["duration_hours"]) < 0:
            reasons.append("отрицательная длительность")
    except (TypeError, ValueError):
        reasons.append("длительность не число")
    if (row["format"] or "").strip() not in {"online", "offline"}:
        reasons.append("неверный формат")
    return reasons


checks = [(r["enrollment_id"], check(r)) for r in enrollments]
bad = [(enrollment_id, reasons) for enrollment_id, reasons in checks if reasons]
good_cnt = len(checks) - len(bad)
print(f"Валидных строк: {good_cnt} из {len(checks)} ({good_cnt / len(checks):.0%})")
for enrollment_id, reasons in bad:
    print(f"  {enrollment_id}: {', '.join(reasons)}")

Валидных строк: 11 из 19 (58%)
  EDU-202609-004: неверный номер заявки, неверный e-mail
  EDU-202610-0008: неверный e-mail
  EDU-202610-0011: курс не из справочника
  EDU-202610-0012: неизвестный статус
  EDU-202610-0013: отрицательная длительность
  edu-202610-0015: неверный номер заявки
  EDU-2026AA-0016: неверный номер заявки
  EDU-202610-0017: лишние колонки, неверный e-mail, курс не из справочника, неизвестный статус, неверная дата, длительность не число, неверный формат


**Интерпретация.** Валидны 11 строк из 19 (58%), уникальных валидных заявок
10. Самая тяжёлая строка - EDU-202610-0017: лишняя запятая внутри адреса
сдвинула все колонки, и строка собрала шесть ошибок сразу. Остальные отказы
точечные: битые номера, адрес без зоны, курс XX-999 вне справочника, статус
paused, отрицательная длительность. Каждая причина понятна и адресна - оператор
может исправить конкретные строки и прислать файл повторно.

## Показатель 3. Заявки по статусам

Дальше все показатели считаются только по уникальным валидным заявкам -
именно они попадут в базу данных.

In [4]:
seen = set()
valid_rows = []
for r in enrollments:
    if not check(r) and r["enrollment_id"] not in seen:
        seen.add(r["enrollment_id"])
        valid_rows.append(r)

by_status = Counter(r["status"] for r in valid_rows)
print(f"Уникальных валидных заявок: {len(valid_rows)}")
for status, cnt in by_status.most_common():
    print(f"  {status}: {cnt}")

Уникальных валидных заявок: 10
  enrolled: 4
  on_review: 3
  completed: 2
  cancelled: 1


**Интерпретация.** Из 10 заявок 4 уже оплачены (`enrolled`), 2 завершены
(`completed`) - это 6 «денежных» заявок. Ещё 3 в статусе `on_review` ждут
ручной проверки оператора, и 1 студент отменил заявку. Поток здоровый:
отказов мало, а половина воронки уже на платном этапе.

## Показатель 4. Заявки по курсам

In [5]:
titles = {c["course_code"]: c["title"] for c in courses}
by_course = Counter(r["course_code"] for r in valid_rows)
for code, cnt in by_course.most_common():
    print(f"  {code} «{titles[code]}»: {cnt} заявок ({cnt / len(valid_rows):.0%})")

  PY-100 «Основы Python»: 5 заявок (50%)
  DA-200 «Анализ данных»: 2 заявок (20%)
  WD-300 «Веб-разработка»: 2 заявок (20%)
  QA-150 «Тестирование ПО»: 1 заявок (10%)


**Интерпретация.** Половина всех заявок (5 из 10) - «Основы Python»:
это входной курс, через него начинают почти все. «Анализ данных» и
«Веб-разработка» взяли по 2 заявки, «Тестирование ПО» - одну. Курс
«Архитектура ПО» (SE-250) в выгрузке не фигурирует вовсе - заявки на него
пока не приходят.

## Показатель 5. Средняя длительность по курсам

In [6]:
by_hours = defaultdict(list)
for r in valid_rows:
    by_hours[r["course_code"]].append(float(r["duration_hours"]))
for code, hours in sorted(by_hours.items()):
    print(f"  {code} «{titles[code]}»: средняя {sum(hours) / len(hours):.1f} ч")

  DA-200 «Анализ данных»: средняя 48.0 ч
  PY-100 «Основы Python»: средняя 36.0 ч
  QA-150 «Тестирование ПО»: средняя 24.0 ч
  WD-300 «Веб-разработка»: средняя 60.0 ч


**Интерпретация.** Длительность внутри курса одинаковая у всех студентов -
это плановый объём программы (24-60 часов). Самый длинный курс -
«Веб-разработка» (60 ч), самый короткий - «Тестирование ПО» (24 ч). Данных о
реальном прогрессе пока нет: показатель подтверждает корректность загрузки,
а не поведение студентов.

## Показатель 6. Потенциальная выручка по активным заявкам

Считаем как цену курса × число заявок в статусах `enrolled` и `completed`
(заявки `on_review` денег пока не принесли, `cancelled` исключены).

In [7]:
prices = {c["course_code"]: int(c["price_rub"]) for c in courses}
paid = [r for r in valid_rows if r["status"] in ("enrolled", "completed")]
revenue = Counter()
for r in paid:
    revenue[r["course_code"]] += prices[r["course_code"]]
total = 0
for code, amount in revenue.most_common():
    total += amount
    print(f"  {code} «{titles[code]}»: {amount} ₽")
print(f"Итого потенциальная выручка: {total} ₽")

  PY-100 «Основы Python»: 180000 ₽
  WD-300 «Веб-разработка»: 84000 ₽
  DA-200 «Анализ данных»: 68000 ₽
Итого потенциальная выручка: 332000 ₽


**Интерпретация.** Активные заявки (6 из 10) несут 332 000 ₽: «Основы Python»
- 180 000 ₽, «Веб-разработка» - 84 000 ₽, «Анализ данных» - 68 000 ₽. Три
заявки в статусе `on_review` - это ещё до 204 000 ₽ («Веб-разработка»
84 000 ₽, «Анализ данных» 68 000 ₽, «Тестирование ПО» 52 000 ₽), если
оператор переведёт их в `enrolled`: скорость проверки напрямую
конвертируется в деньги.

## Показатель 7. Форматы обучения

In [8]:
by_format = Counter(r["format"] for r in valid_rows)
for fmt, cnt in by_format.most_common():
    print(f"  {fmt}: {cnt} заявок ({cnt / len(valid_rows):.0%})")

  online: 6 заявок (60%)
  offline: 4 заявок (40%)


**Интерпретация.** Онлайн выбирают 6 заявок из 10 (60%), офлайн - 4.
Студенты явно предпочитают учиться удалённо; при этом самые дорогие офлайн-
курсы («Веб-разработка», «Анализ данных») пока продаются только в офлайне -
стоит проверить, не теряет ли платформа онлайн-заявки из-за формата по
умолчанию в справочнике курсов.

## Выводы для руководителя

1. **Качество выгрузки - главный операционный риск.** Валидны 58% строк файла;
   из 10 уникальных валидных заявок до базы доходит ровно 10, остальное
   отклоняется с конкретными причинами. Оператору стоит чинить выгрузку:
   дубликат EDU-202610-0006 и строка EDU-202610-0017 со сдвинутыми колонками -
   следствие ручной правки файла.
2. **Портфель заявок небольшой, но здоровый.** 10 заявок: 6 активных
   (enrolled/completed), 3 на проверке, 1 отмена. Воронка не разваливается,
   но объём пока «демо-уровень» - динамику увидим на следующих выгрузках.
3. **Деньги сейчас делает «Основы Python».** Курс собирает 50% заявок и
   180 000 ₽ из 332 000 ₽ потенциальной выручки. Ещё 204 000 ₽
   сидят в трёх заявках `on_review` - быстрая проверка оператором даёт
   +61% к текущей выручке.
4. **Онлайн доминирует (60%), но дорогие курсы пока офлайн.** Если включить
   онлайн-формат для «Веб-разработки» и «Анализа данных», платформа не теряет
   доступность самых дорогих программ.
5. **Курс SE-250 заявок не получил ни одной** - проверить, видно ли его на
   сайте и не ошибся ли оператор в справочнике курсов.